# Taiwan and Philippines: growth accounting

Original 14.05 analysis by Lauren Nichols. Run from this notebooks directory. The residual includes omitted inputs and measurement error; it is not a causal estimate of technology.

In [ ]:
# Install dependencies from requirements.txt before running.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_excel("../data/macro_data.xlsx", sheet_name = 'Data')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- config
COUNTRIES = ["Taiwan", "Philippines"]
YEAR_MIN, YEAR_MAX = 1950, 2023


In [ ]:
data.columns

In [ ]:
def prep_country_stock(df, country):
    cols = ["country", "year", "rgdpe", "rnna", "emp", "labsh", "hc"]
    m = (df.loc[df["country"] == country, cols]
           .sort_values("year")
           .query("@YEAR_MIN <= year <= @YEAR_MAX")
           .dropna(subset=["rgdpe","rnna","emp","labsh", "hc"]))
    # per-worker diagnostics
    m["Y_over_L"] = m["rgdpe"] / m["emp"]
    #used capital stock
    m["K_over_L"] = m["rnna"] / m["emp"]
    m["H_over_L"] = m["hc"]   
    return m

def log_growth(s):
    return np.log(s).diff()

def do_growth_accounting_stock(df_country):
    df = df_country.copy()
    # capital share
    df["alpha"] = 1.0 - df["labsh"]
    # growth rates
    df["gY"] = log_growth(df["rgdpe"])
    df["gK"] = log_growth(df["rnna"])
    df["gL"] = log_growth(df["emp"])
    df["gH"] = log_growth(df["hc"]) 
    # contributions
    df["cap_contrib"]   = df["alpha"] * df["gK"]
   # df["hc_contrib"]   = (1.0 - df["alpha"]) * df["gH"]
    df["labor_contrib"] = (1.0 - df["alpha"]) * df["gL"]
    df["tfp_residual"]  = df["gY"] - df["cap_contrib"] - df["labor_contrib"]
    # per-worker growth diagnostic
    df["gY_over_L"] = log_growth(df["Y_over_L"])
    return df.dropna(subset=["gY","gK","gL", "gH", "alpha"])

def summarize(df_ga):
    return pd.Series({
        "avg_gY": df_ga["gY"].mean(),
        "avg_cap": df_ga["cap_contrib"].mean(),
        #  "avg_hc" : df_ga["hc_contrib"].mean(),
        "avg_labor": df_ga["labor_contrib"].mean(),
        "avg_tfp": df_ga["tfp_residual"].mean(),
        "avg_gY_over_L": df_ga["gY_over_L"].mean(),
        "alpha_avg": df_ga["alpha"].mean()
    })


In [ ]:
def plot_GDP(dfA, dfB, labelA, labelB):
    plt.figure()
    plt.plot(dfA["year"], np.log(dfA["Y_over_L"]), label=f"{labelA}", color="blue")
    plt.plot(dfB["year"], np.log(dfB["Y_over_L"]), label=f"{labelB}", color="red")
    plt.title("Log Real GDP per Worker PPP-adjusted")
    plt.xlabel("Year"); plt.ylabel("Log Real GDP per Worker")
    plt.legend(); plt.tight_layout(); plt.show()

def plot_capital(dfA, dfB, labelA, labelB):
    plt.figure()
    plt.plot(dfA["year"], np.log(dfA["K_over_L"]), label=f"{labelA}", color="blue")
    plt.plot(dfB["year"], np.log(dfB["K_over_L"]), label=f"{labelB}", color = "red")
    plt.title("Log Capital per Worker")
    plt.xlabel("Year"); plt.ylabel("Log Capital per Worker")
    plt.legend(); plt.tight_layout(); plt.show()

def plot_humancapital(dfA, dfB, labelA, labelB):
    plt.figure()
    plt.plot(dfA["year"], np.log(dfA["hc"]), label=f"{labelA}", color="blue")
    plt.plot(dfB["year"], np.log(dfB["hc"]), label=f"{labelB}", color = "red")
    plt.title("Log Human Capital per Worker")
    plt.xlabel("Year"); plt.ylabel("Log Human Capital per Worker")
    plt.legend(); plt.tight_layout(); plt.show()

def plot_labor(dfA, dfB, labelA, labelB):
    plt.figure()
    plt.plot(dfA["year"], np.log(dfA["emp"]), label=f"{labelA}", color="blue")
    plt.plot(dfB["year"], np.log(dfB["emp"]), label=f"{labelB}", color = "red")
    plt.title("log(labor)")
    plt.xlabel("Year"); plt.ylabel("Log labor")
    plt.legend(); plt.tight_layout(); plt.show()

def plot_productivity(dfA, dfB, labelA, labelB):
    plt.figure()
    plt.plot(dfA["year"], dfA["tfp_residual"], label=f"{labelA}", color="blue")
    plt.plot(dfB["year"], dfB["tfp_residual"], label=f"{labelB}", color = "red")
    plt.title("Residual productivity growth")
    plt.xlabel("Year"); plt.ylabel("Annual growth (log points)")
    plt.legend(); plt.tight_layout(); plt.show() 
    
def plot_contrib(df_ga, title):
    plt.figure()
    plt.plot(df_ga["year"], df_ga["cap_contrib"], label="Capital")
    plt.plot(df_ga["year"], df_ga["labor_contrib"], label="Labor")
    plt.plot(df_ga["year"], df_ga["tfp_residual"], label="TFP (residual)")
    plt.title(f"Growth Contributions — {title}")
    plt.xlabel("Year"); plt.ylabel("Contribution (log points)")
    plt.legend(); plt.tight_layout(); plt.show()

def plot_avg(summary):
    idx = np.arange(len(summary))
    width = 0.25
    plt.figure()
    plt.bar(idx - width, summary["avg_cap"].values, width, label="Capital")
    plt.bar(idx,         summary["avg_labor"].values, width, label="Labor")
    plt.bar(idx + width, summary["avg_tfp"].values, width, label="TFP")
    plt.xticks(idx, summary.index)
    plt.ylabel("Avg annual contribution (log points)")
    plt.title("Average Contributions to Output Growth")
    plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# assumes you already loaded your data
# data = pd.read_excel("../data/macro_data.xlsx", sheet_name="Data")

country_frames = {}
ga_frames = {}
summ_rows = []

for c in COUNTRIES:
    dfc = prep_country_stock(data, c)
    country_frames[c] = dfc
    gac = do_growth_accounting_stock(dfc)
    ga_frames[c] = gac
    s = summarize(gac); s.name = c; summ_rows.append(s)

summary = pd.DataFrame(summ_rows)
summary


In [ ]:
plot_productivity(ga_frames[COUNTRIES[0]], ga_frames[COUNTRIES[1]],
            COUNTRIES[0], COUNTRIES[1])


In [ ]:
# GDP per worker
plot_GDP(country_frames[COUNTRIES[0]], country_frames[COUNTRIES[1]],
            COUNTRIES[0], COUNTRIES[1])
plot_capital(country_frames[COUNTRIES[0]], country_frames[COUNTRIES[1]],
            COUNTRIES[0], COUNTRIES[1])
plot_labor(country_frames[COUNTRIES[0]], country_frames[COUNTRIES[1]],
            COUNTRIES[0], COUNTRIES[1])
plot_humancapital(country_frames[COUNTRIES[0]], country_frames[COUNTRIES[1]],
            COUNTRIES[0], COUNTRIES[1])

# Contributions over time
for c in COUNTRIES:
    plot_contrib(ga_frames[c], c)

# Average contributions bar chart
plot_avg(summary[["avg_cap","avg_labor","avg_tfp"]])


In [ ]:
# --- 5-year averages for stacked bar chart
def five_year_summary(df_ga):
    df = df_ga.copy()
    df["period"] = (df["year"] // 5) * 5   # e.g. 1961–65 -> 1960
    grouped = df.groupby("period")[["cap_contrib","labor_contrib", "tfp_residual"]].mean()
    return grouped

# compute for each country
fiveyr_summaries = {}
for c in COUNTRIES:
    fiveyr_summaries[c] = five_year_summary(ga_frames[c])

# --- stacked bar chart
def plot_stacked(summary, country):
    plt.figure()
    bottom = np.zeros(len(summary))
    for comp, color in zip(["cap_contrib","labor_contrib", "tfp_residual"],
                           ["#d73027","#1a9850","#4575b4"]):  # red, green, blue
        plt.bar(summary.index, summary[comp], bottom=bottom, label=comp, color=color)
        bottom += summary[comp].values
    plt.title(f"Growth Accounting by 5-Year Period — {country}")
    plt.xlabel("Period")
    plt.ylabel("Avg annual growth contribution")
    plt.legend()
    plt.tight_layout()
    plt.show()

for c in COUNTRIES:
    plot_stacked(fiveyr_summaries[c], c)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def periodize(df_ga: pd.DataFrame, bin_size: int = 5) -> pd.DataFrame:
    """
    Collapse annual growth-accounting rows into bin_size-year averages.
    Returns a table with columns: cap_contrib, labor_contrib, tfp_residual, gY
    Index are period labels like '1960–1964' or '1970–1979'.
    """
    df = df_ga.copy()
    if "year" not in df:
        raise ValueError("df_ga must contain a 'year' column")

    # anchor bins to decade starts (…50, …60, etc.)
    start_anchor = (int(df["year"].min()) // bin_size) * bin_size
    # start of each period
    df["period_start"] = start_anchor + ((df["year"] - start_anchor) // bin_size) * bin_size
    df["period_end"]   = df["period_start"] + (bin_size - 1)

    # nice label like 1960–1964
    df["period"] = df["period_start"].astype(int).astype(str) + "–" + df["period_end"].astype(int).astype(str)

    grouped = (df.groupby("period")[["cap_contrib","labor_contrib","tfp_residual","gY"]]
                 .mean()
                 .sort_index())
    return grouped

def align_periods(left: pd.DataFrame, right: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Reindex both summaries to the union of period labels so the x-axes match."""
    all_periods = sorted(set(left.index).union(set(right.index)))
    return (left.reindex(all_periods).fillna(0.0),
            right.reindex(all_periods).fillna(0.0))


In [ ]:
def plot_two_country_stacked(summary_left: pd.DataFrame,
                             summary_right: pd.DataFrame,
                             name_left: str, name_right: str):
    L, R = align_periods(summary_left, summary_right)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    comp_cols   = ["cap_contrib","labor_contrib","tfp_residual"]
    comp_labels = ["Capital", "Labor", "TFP"]
    comp_colors = {
        "cap_contrib":   "#0E4C92",  # dark blue
        "labor_contrib": "#E67E22",  # orange
        "tfp_residual":  "#1f77b4"   # blue-gray
    }

    def _stack(ax, tab, title, show_left_ticks=True, show_right_ticks=False):
        periods = tab.index.tolist()
        x = np.arange(len(periods))
        width = 0.85
        bottom_pos = np.zeros(len(periods))
        bottom_neg = np.zeros(len(periods))

        for col, lab in zip(comp_cols, comp_labels):
            vals = tab[col].values
            pos = np.where(vals >= 0, vals, 0.0)
            neg = np.where(vals <  0, vals, 0.0)

            ax.bar(x, pos, width=width, bottom=bottom_pos,
                   color=comp_colors[col], edgecolor="white",
                   linewidth=0.6, label=lab, alpha=0.9)
            bottom_pos += pos

            ax.bar(x, neg, width=width, bottom=bottom_neg,
                   color=comp_colors[col], edgecolor="white",
                   linewidth=0.6, alpha=0.9)
            bottom_neg += neg

        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(periods, rotation=45, ha="right")
        ax.set_xlabel("Period")
        ax.grid(axis="y", linestyle=":", alpha=0.35)

        # show both left and right ticks if requested
        if show_left_ticks:
            ax.set_ylabel("Avg annual growth contribution")
            ax.tick_params(axis="y", labelleft=True, left=True)
        if show_right_ticks:
           # ax.set_ylabel("Avg annual growth contribution")
            ax.yaxis.set_label_position("right")
            ax.yaxis.tick_right()
            ax.tick_params(axis="y", labelright=True, right=True)

    # left plot: normal ticks
    _stack(axes[0], L, name_left, show_left_ticks=True, show_right_ticks=False)
    # right plot: also show tick labels/numbers on right
    _stack(axes[1], R, name_right, show_left_ticks=False, show_right_ticks=True)

    # legend
    handles, labels = axes[0].get_legend_handles_labels()
    bylab = dict(zip(labels, handles))
    fig.legend(bylab.values(), bylab.keys(), loc="lower center", ncol=3,
               bbox_to_anchor=(0.5, -0.03), frameon=False)
    fig.suptitle("Growth Accounting by Period", y=1.04, fontsize=12)
    fig.tight_layout()
    plt.show()


In [ ]:
BIN = 5      # set to 5 or 10

cA, cB = COUNTRIES[0], COUNTRIES[1]

sumA = periodize(ga_frames[cA], bin_size=BIN)
sumB = periodize(ga_frames[cB], bin_size=BIN)

plot_two_country_stacked(sumA, sumB, cA, cB)


In [ ]:
# turn a periodized summary (from periodize) into the paper-ready table
def format_table_from_periodized(pz: pd.DataFrame, add_check: bool = True) -> pd.DataFrame:
    # pz is what periodize() returns: columns cap_contrib, labor_contrib, hc_contrib, tfp_residual, gY
    cols_needed = ["gY","cap_contrib","labor_contrib","tfp_residual"]
    missing = [c for c in cols_needed if c not in pz.columns]
    if missing:
        raise ValueError(f"periodized df missing columns: {missing}")

    tab = (pz[cols_needed]
           .mul(100.0)  # -> percentage points per year
           .rename(columns={
               "gY": "gy",
               "cap_contrib": "αgk",
               "labor_contrib": "(1-α)gL",
               "tfp_residual": "gA"
           })
           .reindex(columns=["gy","αgk","(1-α)gL","gA"])
           .round(1))

    if add_check:
        tab["check"] = (tab["gy"] - tab[["αgk","(1-α)gL","gA"]].sum(axis=1)).round(1)

    return tab

# combine two country tables side-by-side
def two_country_table_from_periodized(pz_left: pd.DataFrame, pz_right: pd.DataFrame,
                                      name_left: str, name_right: str, drop_check: bool = True) -> pd.DataFrame:
    L = format_table_from_periodized(pz_left, add_check=True)
    R = format_table_from_periodized(pz_right, add_check=True)

    # align periods
    all_periods = sorted(set(L.index) | set(R.index))
    L, R = L.reindex(all_periods), R.reindex(all_periods)

    if drop_check:
        L = L.drop(columns=["check"], errors="ignore")
        R = R.drop(columns=["check"], errors="ignore")

    L.columns = pd.MultiIndex.from_product([[name_left], L.columns])
    R.columns = pd.MultiIndex.from_product([[name_right], R.columns])
    return pd.concat([L, R], axis=1)


In [ ]:
table_2x = two_country_table_from_periodized(sumA, sumB, cA, cB)

# print(table_2x.to_string())             # console
table_2x.to_csv("../results/growth_table_5yr.csv")
table_2x.to_latex("../results/growth_table_5yr.tex", multicolumn=True, multirow=True)